# Medical Charges Prediction

An end-to-end regression project that estimates annual medical insurance charges from age, BMI, number of children, smoking status, sex, and region. The workflow follows the provided MediCharge reference and the repository’s Cardiovascular Disease project structure. This is an educational estimate, not a quote or medical/financial advice.

## 1. Load libraries and data

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

ROOT = Path.cwd()
if not (ROOT / "data" / "insurance.csv").exists():
    ROOT = Path("..")
DATA_PATH = ROOT / "data" / "insurance.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
df.head()

## 2. Explore and clean

In [ ]:
print("Shape:", df.shape)
display(df.describe(include="all").T)
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
assert set(["age", "sex", "bmi", "children", "smoker", "region", "charges"]).issubset(df.columns)
df["risk"] = ((df["bmi"] >= 30) & (df["smoker"].str.lower() == "yes")).astype(int)
df["risk"].value_counts()

In [ ]:
numeric_cols = ["age", "bmi", "children", "risk"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, ax in zip(["age", "bmi", "children", "charges"], axes.flat):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=.7, ax=axes[0])
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", alpha=.7, ax=axes[1])
plt.tight_layout()

## 3. Train a preprocessing + regression pipeline

The engineered `risk` feature flags policyholders with BMI ≥ 30 who smoke. Numeric columns use median imputation and robust scaling; categorical columns use most-frequent imputation and one-hot encoding. The preprocessing is fitted only on training folds to avoid leakage.

In [ ]:
X = df.drop(columns="charges")
y = df["charges"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True
)
num_cols = ["age", "bmi", "children", "risk"]
cat_cols = ["sex", "smoker", "region"]
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))]), cat_cols),
])
pipeline = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])
search = GridSearchCV(pipeline, {"model__fit_intercept": [True, False]}, cv=5, scoring="r2", n_jobs=-1)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best params:", search.best_params_)
print("Mean CV R²:", round(search.best_score_, 4))

## 4. Evaluate held-out performance

In [ ]:
def regression_metrics(model, X, y):
    pred = model.predict(X)
    return {"MAE": mean_absolute_error(y, pred),
            "RMSE": np.sqrt(mean_squared_error(y, pred)),
            "R²": r2_score(y, pred)}

train_metrics = regression_metrics(best_model, X_train, y_train)
test_metrics = regression_metrics(best_model, X_test, y_test)
print("Train:", train_metrics)
print("Test: ", test_metrics)
print("CV R²:", search.best_score_)
y_pred = best_model.predict(X_test)
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=.65)
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(limits, limits, "--", color="black")
plt.xlabel("Actual charges ($ / year)")
plt.ylabel("Predicted charges ($ / year)")
plt.title("Actual vs. predicted charges")
plt.tight_layout()

## 5. Save the fitted pipeline

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "medical_charges_pipeline.joblib"
joblib.dump(best_model, MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")